# Mental Health Support Chatbot — Fine-Tuned DistilGPT2

## Problem Statement
Mental health conversations require empathy, patience, and emotional attunement — qualities generic language models lack. When someone says *"I feel overwhelmed by work"*, a standard LM might respond with generic advice or an irrelevant tangent, missing the emotional cue entirely.

## Goal
Fine-tune **DistilGPT2** on the **EmpatheticDialogues** dataset to produce a chatbot that recognizes emotional context and responds with a supportive, empathetic tone — specifically targeting stress and anxiety scenarios.

## Skills Demonstrated
- Causal language model fine-tuning with Hugging Face Transformers
- Dataset preprocessing & tokenization for conversational AI
- Training with `Trainer` API & `TrainingArguments`
- Qualitative before/after model comparison
- checkpoints & Colab-disconnect-safe training practices

---
## 1. SETUP & GPU CHECK

In [ ]:
# Install required libraries inside Colab
!pip install -q transformers datasets torch accelerate

In [ ]:
import torch
import transformers
import datasets
import accelerate

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Datasets version: {datasets.__version__}")
print(f"Accelerate version: {accelerate.__version__}")

# Check GPU — critical for DistilGPT2 fine-tuning
if torch.cuda.is_available():
    print(f"\nGPU available: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("\n*** WARNING: No GPU detected! ***")
    print("Fine-tuning DistilGPT2 will be extremely slow or crash on CPU.")
    print("Please enable a GPU runtime in Colab: Runtime > Change runtime type > T4 GPU")
    print("Stopping execution — enable GPU and re-run this cell.")

---
## 2. LOAD DATASET

In [ ]:
from datasets import load_dataset

# Load the EmpatheticDialogues dataset from Hugging Face
# This dataset contains ~25k conversations grounded in emotional situations
dataset = load_dataset("facebook/empathetic_dialogues", trust_remote_code=True)

print(f"Dataset structure: {dataset}")
print(f"\nNumber of training samples: {len(dataset['train'])}")
print(f"Number of validation samples: {len(dataset['validation'])}")

In [ ]:
# Peek at a few sample conversations to understand the format
for i, sample in enumerate(dataset['train'].select(range(3))):
    print(f"=== Sample {i+1} ===")
    print(f"Situation: {sample['prompt']}")
    print(f"Emotion: {sample['context']}")
    print(f"Utterance: {sample['utterance']}")
    print(f"Response: {sample['utterance']}")  # Note: EmpatheticDialogues has 'utterance' as the prompt
    print()

### What is EmpatheticDialogues?

[EmpatheticDialogues](https://arxiv.org/abs/1811.00207) (Rashkin et al., 2019) is a collection of **~25k one-sided conversations** where a *speaker* describes an emotional situation, and a *listener* responds empathetically. Each conversation is labeled with one of 32 emotions (sad, anxious, grateful, etc.).

**Why it's suited for this task:**
- The listener responses are written to be empathetic and emotionally aware — exactly the tone we want our chatbot to learn.
- The emotion labels help the model understand which emotional context it's responding to.
- The conversational format (situation → utterance → response) maps naturally to a prompt-response fine-tuning task.

---
## 3. PREPROCESS DATA

In [ ]:
from transformers import AutoTokenizer

# Load DistilGPT2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

# CRITICAL: DistilGPT2 has NO default pad_token (unlike BERT/GPT-2 which uses eos).
# We need a pad_token so the Trainer can pad batches to equal length.
# Without it, the collation step crashes when sequences differ in length.
# Standard practice: set pad_token = eos_token (end-of-sequence token).
tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded: {tokenizer}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")
print(f"EOS token: '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")

In [ ]:
# Format each conversation into a prompt-response text for causal LM training.
# We use a structured template so the model learns:
#   1. What the user said (User: ...)
#   2. What an empathetic response looks like (Response: ...)
# The model will learn to continue the text after \"Response: \"

def format_conversation(examples):
    """Convert dataset examples into a single text string for causal LM."""
    texts = []
    for situation, utterance, response in zip(
        examples['prompt'], examples['utterance'], examples['utterance']
    ):
        # Handle None situations gracefully
        sit = situation if situation else "[No context]"
        # Format: Context + User utterance + Response
        text = f"Context: {sit}\nUser: {utterance}\nResponse: {response}"
        texts.append(text)
    return {"text": texts}

# EmpatheticDialogues only has 'utterance' (user prompt) and no separate response column.
# Actually, looking more carefully: the dataset has both 'utterance' and 'utterance' is just the prompt.
# Wait — let's check the actual column names.
print(dataset['train'].column_names)

In [ ]:
# The dataset has 'utterance' as the user line. For empathetic response training,
# we pair consecutive utterance lines (or better: pair each utterance with
# what the dataset provides). Let's inspect the actual conversation structure.

# EmpatheticDialogues: each row has 'utterance' (what the speaker says).
# For fine-tuning a chatbot to RESPOND empathetically, we need to teach
# it the pattern: hear user -> give empathetic reply.
# The dataset's 'utterance' column IS already the human response in context.
# But actually — looking at the dataset structure, each conversation
# in EmpatheticDialogues has turns. Let's build proper prompt-response pairs
# by pairing odd/even lines within each conversation_id.

from collections import defaultdict

# Build prompt-response pairs from conversation turns
def build_prompt_response_pairs(dataset_split):
    """
    EmpatheticDialogues stores conversation_id so we can reconstruct
    the full dialogue. Each conversation_id has multiple turns.
    We pair each utterance (as the user prompt) with the NEXT utterance
    (as the empathetic response).
    """
    convs = defaultdict(list)
    for sample in dataset_split:
        convs[sample['conv_id']].append({
            'situation': sample['prompt'],
            'utterance': sample['utterance'],
        })
    
    pairs = []
    for conv_id, turns in convs.items():
        situation = turns[0]['prompt'] if turns[0]['prompt'] else ""
        for i in range(len(turns) - 1):
            prompt = turns[i]['utterance']
            response = turns[i + 1]['utterance']
            pairs.append({
                'situation': situation,
                'prompt': prompt,
                'response': response,
            })
    return pairs

train_pairs = build_prompt_response_pairs(dataset['train'])
val_pairs = build_prompt_response_pairs(dataset['validation'])

print(f"Built {len(train_pairs)} training prompt-response pairs")
print(f"Built {len(val_pairs)} validation prompt-response pairs")
print(f"\nExample pair:")
print(f"  Situation: {train_pairs[0]['situation']}")
print(f"  Prompt: {train_pairs[0]['prompt']}")
print(f"  Response: {train_pairs[0]['response']}")

In [ ]:
# Now format pairs into a text string for causal LM
def format_pairs(pairs):
    texts = []
    for p in pairs:
        sit = p['situation'] if p['situation'] else "[No context]"
        text = f"Context: {sit}\nUser: {p['prompt']}\nResponse: {p['response']}"
        texts.append(text)
    return texts

train_texts = format_pairs(train_pairs)
val_texts = format_pairs(val_pairs)

print("=== Formatted example ===")
print(train_texts[0])
print("\n" + "=" * 50)
print(train_texts[1])

In [ ]:
# Tokenize all texts.
# max_length=128: keeps sequences short (reduces memory/compute).
#   Most conversations in this dataset are 1-3 sentences, well under 128 tokens.
#   Truncating at 128 is safe and speeds up training significantly.
# truncation=True: cuts texts longer than max_length so batches stay uniform.
# padding=False: we let the data collator handle padding during training.

def tokenize_function(texts):
    return tokenizer(
        texts,
        max_length=128,
        truncation=True,
        padding=False,          # Trainer handles dynamic padding
        return_tensors=None,    # return lists (Trainer will convert)
    )

train_tokenized = tokenize_function(train_texts)
val_tokenized = tokenize_function(val_texts)

print(f"Training tokenized samples: {len(train_tokenized['input_ids'])}")
print(f"Validation tokenized samples: {len(val_tokenized['input_ids'])}")

In [ ]:
# Decode one tokenized example back to text to verify format is correct
example_idx = 0
decoded = tokenizer.decode(train_tokenized['input_ids'][example_idx], skip_special_tokens=True)
print("=== Tokenized example (decoded back) ===")
print(decoded)
print("\n" + "=" * 50)
print(f"Token count: {len(train_tokenized['input_ids'][example_idx])} tokens")

In [ ]:
# Wrap tokenized data into a torch Dataset for the Trainer API
from torch.utils.data import Dataset

class EmpatheticDataset(Dataset):
    def __init__(self, input_ids, attention_mask):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],  # DataCollator handles tensor creation + padding
            "attention_mask": self.attention_mask[idx],
            "labels": self.input_ids[idx],  # causal LM: labels = input_ids
        }

train_dataset = EmpatheticDataset(train_tokenized['input_ids'], train_tokenized['attention_mask'])
val_dataset = EmpatheticDataset(val_tokenized['input_ids'], val_tokenized['attention_mask'])

print(f"Train dataset size: {len(train_dataset)} samples")
print(f"Validation dataset size: {len(val_dataset)} samples")

---
## 4. LOAD PRETRAINED MODEL

In [ ]:
from transformers import AutoModelForCausalLM

# Load DistilGPT2 — a distilled (smaller, faster) version of GPT-2.
# Why DistilGPT2 instead of GPT-2?
#   - 82M parameters vs GPT-2's 124M — trains faster, uses less GPU memory
#   - Retains ~97% of GPT-2's language understanding
#   - Ideal for an empathetic chatbot where we want quality + speed
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

# Count total parameters to verify model size
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: DistilGPT2")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Memory estimate: {total_params * 4 / 1e6:.1f} MB (fp32)")

# Move model to GPU for training
if torch.cuda.is_available():
    model = model.to("cuda")
    print(f"Model moved to: {next(model.parameters()).device}")
else:
    print("Model staying on CPU (training will be slow)")

---
## 5. FINE-TUNE USING Trainer API


In [ ]:
from transformers import TrainingArguments, Trainer
from torch.nn.utils.rnn import pad_sequence

# Custom collator: dynamically pad each batch to the longest sequence.
# We avoid DataCollatorForLanguageModeling here because Transformers 5.x
# changed its internals -- this manual approach is simpler and reliable.
def data_collator(features):
    batch = {}
    batch["input_ids"] = pad_sequence(
        [torch.tensor(f["input_ids"], dtype=torch.long) for f in features],
        batch_first=True, padding_value=tokenizer.pad_token_id
    )
    batch["attention_mask"] = pad_sequence(
        [torch.tensor(f["attention_mask"], dtype=torch.long) for f in features],
        batch_first=True, padding_value=0
    )
    batch["labels"] = pad_sequence(
        [torch.tensor(f["labels"], dtype=torch.long) for f in features],
        batch_first=True, padding_value=-100  # -100 ignores padding in loss calc
    )
    return batch

training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy="epoch",
    logging_steps=50,
    eval_strategy="epoch",
    report_to="none",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

print("Data collator & training args ready")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

# Kick off fine-tuning
train_result = trainer.train()

In [ ]:
# Extract and display training/validation loss from Trainer log history
logs = trainer.state.log_history

print("=== Training Loss Progression ===")
for entry in logs:
    if 'loss' in entry and 'eval_loss' not in entry:
        print(f"Step {entry['step']:>5d}: loss = {entry['loss']:.4f}")

print("\n=== Validation Loss Progression ===")
for entry in logs:
    if 'eval_loss' in entry:
        print(f"Epoch end: eval_loss = {entry['eval_loss']:.4f}")

---
## 6. BEFORE vs AFTER COMPARISON


In [ ]:
from transformers import AutoModelForCausalLM

# Load a SEPARATE copy of the original (non-fine-tuned) DistilGPT2 for comparison.
# The 'model' variable now holds the fine-tuned version, so we need a fresh one.
original_model = AutoModelForCausalLM.from_pretrained("distilgpt2")
if torch.cuda.is_available():
    original_model = original_model.to("cuda")

# Test prompts: realistic stress/anxiety scenarios
test_prompts = [
    "Context: I've been feeling really anxious about my upcoming exam\nUser: I can't stop worrying that I'll fail\nResponse:",
    "Context: I feel like I'm not good enough at my job\nUser: Everyone else seems to have it together except me\nResponse:",
    "Context: I had a fight with my best friend yesterday\nUser: I don't know if our friendship will survive this\nResponse:",
    "Context: I've been feeling lonely since I moved to a new city\nUser: I miss my old friends and haven't made new ones yet\nResponse:",
]

def generate_response(model, prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=" * 60)
print("ORIGINAL DistilGPT2 Responses (before fine-tuning)")
print("=" * 60 + "\n")
for i, prompt in enumerate(test_prompts, 1):
    response = generate_response(original_model, prompt)
    print(f"--- Prompt {i} ---")
    print(f"Response: {response}")
    print()

In [ ]:
print("=" * 60)
print("FINE-TUNED DistilGPT2 Responses")
print("=" * 60 + "\n")
for i, prompt in enumerate(test_prompts, 1):
    response = generate_response(model, prompt)
    print(f"--- Prompt {i} ---")
    print(f"Response: {response}")
    print()

print("=" * 60)
print("BEFORE vs AFTER (direct comparison)")
print("=" * 60)
for i, prompt in enumerate(test_prompts, 1):
    orig = generate_response(original_model, prompt)
    fine = generate_response(model, prompt)
    print(f"\nPrompt {i}:")
    print(f"  BEFORE: {orig}")
    print(f"  AFTER:  {fine}")

### Qualitative Summary

The fine-tuned model should show:
- **More empathetic** responses that acknowledge the user's feelings
- **Less generic** text — fewer off-topic tangents
- **Context-aware** replies that reference the specific situation
- **Warmer tone** compared to the original model's neutral/random completions

If improvements are subtle after only 3 epochs, that's normal — empathetic response generation is nuanced and benefits from more training or a larger model.

---
## 7. SAVE THE MODEL


In [ ]:
import os

# Save model and tokenizer to the project directory.
# From notebook dir (Task5_Mental_Health_Chatbot/notebook/), this resolves to
# Task5_Mental_Health_Chatbot/model/fine_tuned_model/
save_dir = "../../model/fine_tuned_model"
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Model saved to: {os.path.abspath(save_dir)}")
print(f"Files:")
for f in sorted(os.listdir(save_dir)):
    size = os.path.getsize(os.path.join(save_dir, f))
    print(f"  {f}  ({size/1024:.1f} KB)")

In [ ]:
# == COLAB USERS ==
# Colab storage is TEMPORARY — download or save to Drive immediately after training.
# Uncomment and run ONE of these options:

# Option 1: Download ZIP to your local machine
# !zip -r /content/fine_tuned_model.zip /content/fine_tuned_model/
# from google.colab import files
# files.download("/content/fine_tuned_model.zip")

# Option 2: Save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/fine_tuned_model /content/drive/MyDrive/

# == LOCAL USERS ==
# The model is already saved to the path above.
# To create a portable archive:
#   tar -czf fine_tuned_model.tar.gz -C ../../model/fine_tuned_model .

print(f"Model directory: {os.path.abspath(save_dir)}")

---
## 8. SAFETY NOTE

> **IMPORTANT: This chatbot is NOT a replacement for professional mental health support.**

This model is fine-tuned to respond with empathy and emotional awareness, but:
- It has **no understanding of crisis situations** and may give inappropriate responses
- It should **never** be used as a substitute for therapy, counseling, or emergency services
- If someone expresses suicidal ideation, self-harm, or a crisis, the system **must** redirect them to professional resources (e.g., 988 Suicide & Crisis Lifeline)

**What's coming in the next phase (Chat Interface):**
- Crisis keyword detection using regex/keyword matching
- Automatic redirection to crisis resources
- A Gradio chat interface for interacting with the model

*If you or someone you know is struggling, please reach out to a mental health professional or call 988 (US).*